# 03 — Valuation Pipeline Validation

End-to-end check of the valuation modules:
1. Generate data + price scenarios
2. Pick a field-host pair, show capped production
3. Compute NPV array, inspect distributions
4. Run LSMC, show exercise timing and ROV
5. Compare ROV vs static NPV (optionality value)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.config import load_config
from src.data_generation.generator import generate_data
from src.data_generation.profiles import logistic_watercut
from src.price.schwartz_smith import generate_price_scenarios
from src.models import PortfolioAllocation
from src.valuation.capacity import compute_capped_production
from src.valuation.npv import compute_field_npvs
from src.valuation.lsmc import lsmc_field

config = load_config('../config/default.yaml')
print(f"Dimensions: {config['n_fields']} fields, {config['n_hosts']} hosts, {config['n_years']} years")
print(f"Scenarios: {config['price']['n_scenarios']}")

## 1. Generate data and prices

In [ ]:
field_data, host_data, cost_data = generate_data(config)
prices = generate_price_scenarios(config)

# Assemble CAPEX matrix using data-generation n_wells (for display)
capex_matrix = cost_data.capex(field_data.n_wells)

print("=== Fields ===")
for i in range(field_data.n_fields):
    print(f"  {field_data.names[i]}: peak oil={field_data.oil[i].max():.2f} MSm3/y, "
          f"UR={field_data.ultimate_recovery[i]:.1f} MSm3, "
          f"wells={field_data.n_wells[i]}, depth={field_data.depth[i]:.0f}m")

print("\n=== Hosts ===")
for i in range(host_data.n_hosts):
    print(f"  {host_data.names[i]}: oil cap={host_data.oil_capacity[i, 0]:.2f} MSm3/y, "
          f"OPEX={cost_data.opex_rate[i]:.0f} MNOK/y, "
          f"var OPEX={cost_data.variable_opex_rate[i]:.1f} USD/bbl, "
          f"ABEX={cost_data.abex[i]:.0f} MNOK")

print("\n=== Tieback CAPEX (MNOK, cheapest host per field) ===")
for i in range(field_data.n_fields):
    j = capex_matrix[i].argmin()
    print(f"  {field_data.names[i]} -> {host_data.names[j]}: {capex_matrix[i, j]:.0f}")

print(f"\n=== Prices (year 0 mean) ===")
print(f"  Oil: {prices.oil[:, 0].mean():.1f} USD/bbl")
print(f"  Gas: {prices.gas[:, 0].mean():.2f} USD/MMBtu")

## 2. Capacity capping for a test allocation

Each field is assigned to its nearest host, with start years staggered across the planning horizon.

In [ ]:
from scipy.spatial.distance import cdist

# Assign each field to its nearest host
dist_matrix = cdist(field_data.coords, host_data.coords)
assignments = dist_matrix.argmin(axis=1)

# Stagger start years evenly across the first half of the horizon
max_start = config['n_years'] // 2
start_years = np.linspace(0, max_start, field_data.n_fields, dtype=int)

allocation = PortfolioAllocation(
    assignment=assignments,
    start_year=start_years,
    n_wells=field_data.n_wells.copy(),
)

print("Test allocation (nearest-host, staggered start):")
for i in range(allocation.n_fields):
    j = int(allocation.assignment[i])
    d = dist_matrix[i, j]
    print(f"  {field_data.names[i]} → {host_data.names[j]} "
          f"(dist={d:.0f} km, CAPEX={capex_matrix[i, j]:.0f} MNOK, start yr {allocation.start_year[i]})")

# Compute capping parameters
per_well_rate = config['data_generation']['per_well_rate']
n_f = field_data.n_fields
n_y = config['n_years']

max_rates = np.array([
    int(allocation.n_wells[f]) * per_well_rate * field_data.complexity_factor[f]
    for f in range(n_f)
])

t = np.arange(n_y, dtype=float)
wc_fractions = np.zeros((n_f, n_y))
for f in range(n_f):
    wc_fractions[f] = logistic_watercut(
        t, float(field_data.wc_midpoint[f]),
        float(field_data.wc_initial[f]),
        float(field_data.wc_final[f]),
        float(field_data.wc_steepness[f]),
    )

capped = compute_capped_production(
    field_data.oil, host_data, allocation, n_y,
    max_rates=max_rates,
    ramp_years=field_data.ramp_years,
    gor=field_data.gor,
    wc_fractions=wc_fractions,
)

# Pick a host with multiple fields for the capping plots
plot_host = max(capped.keys(), key=lambda h: len(capped[h]['field_indices']))
host_result = capped[plot_host]
print(f"\nPlotting Host {plot_host} ({host_data.names[plot_host]}) — "
      f"{len(host_result['field_indices'])} fields assigned")

years = np.arange(config['n_years'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original (shifted) vs capped oil
ax = axes[0]
for local_i, global_i in enumerate(host_result['field_indices']):
    start = int(allocation.start_year[global_i])
    shifted = np.zeros(config['n_years'])
    usable = min(field_data.n_years, config['n_years'] - start)
    shifted[start:start + usable] = field_data.oil[global_i, :usable]

    ax.plot(years, shifted, '--', alpha=0.5, label=f'{field_data.names[global_i]} original')
    ax.plot(years, host_result['oil'][local_i], '-', label=f'{field_data.names[global_i]} capped')

ax.plot(years, host_data.oil_capacity[plot_host], 'k--', lw=2, label='Host capacity')
ax.set_xlabel('Year')
ax.set_ylabel('Oil production (MSm3/y)')
ax.set_title(f'Host {plot_host} — Oil: Original vs Capped')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Total demand vs capacity
ax = axes[1]
total_original = np.zeros(config['n_years'])
total_capped = host_result['oil'].sum(axis=0)
for local_i, global_i in enumerate(host_result['field_indices']):
    start = int(allocation.start_year[global_i])
    shifted = np.zeros(config['n_years'])
    usable = min(field_data.n_years, config['n_years'] - start)
    shifted[start:start + usable] = field_data.oil[global_i, :usable]
    total_original += shifted

ax.fill_between(years, 0, total_original, alpha=0.3, label='Total demand (uncapped)')
ax.plot(years, total_capped, 'b-', lw=2, label='Total after capping')
ax.plot(years, host_data.oil_capacity[plot_host], 'k--', lw=2, label='Host capacity')
ax.set_xlabel('Year')
ax.set_ylabel('Oil production (MSm3/y)')
ax.set_title(f'Host {plot_host} — Total Demand vs Capacity')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Spatial view of the test allocation

Shows field and host positions with assigned tiebacks highlighted. Line labels show distance (km) and CAPEX (MNOK).

In [ ]:
from scipy.spatial.distance import cdist

dist_matrix = cdist(field_data.coords, host_data.coords)
area = config['data_generation']['area_size_km']

# One color per host for the tieback lines
host_colors = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'purple']

fig, ax = plt.subplots(figsize=(8, 8))

# Draw assigned tiebacks as bold colored lines
for i in range(allocation.n_fields):
    j = int(allocation.assignment[i])
    if j < 0:
        continue
    fx, fy = field_data.coords[i]
    hx, hy = host_data.coords[j]
    d = dist_matrix[i, j]
    c = capex_matrix[i, j]
    color = host_colors[j % len(host_colors)]

    ax.plot([fx, hx], [fy, hy], '-', color=color, linewidth=2.5, alpha=0.8)

    # Label at midpoint
    mx, my = (fx + hx) / 2, (fy + hy) / 2
    ax.annotate(f'{d:.0f} km\n{c:.0f} MNOK', (mx, my), fontsize=7,
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, alpha=0.85))

# Plot fields
ax.scatter(field_data.coords[:, 0], field_data.coords[:, 1],
           s=120, c='steelblue', marker='o', zorder=5, edgecolors='white', linewidth=1.5)
for i, name in enumerate(field_data.names):
    ax.annotate(name, field_data.coords[i], fontsize=9, fontweight='bold',
                textcoords='offset points', xytext=(8, 8), color='steelblue')

# Plot hosts
ax.scatter(host_data.coords[:, 0], host_data.coords[:, 1],
           s=200, c='tomato', marker='s', zorder=5, edgecolors='white', linewidth=1.5)
for j, name in enumerate(host_data.names):
    ax.annotate(name, host_data.coords[j], fontsize=9, fontweight='bold',
                textcoords='offset points', xytext=(8, 8), color='tomato')

ax.set_xlim(-10, area + 10)
ax.set_ylim(-10, area + 10)
ax.set_xlabel('x (km)')
ax.set_ylabel('y (km)')
ax.set_title('Test Allocation — Tieback Assignments')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. NPV distribution

Compute the NPV array for the first field on the selected host. Show how NPV varies across scenarios and exercise years.

In [ ]:
v = config['valuation']

# First field on the selected host
field_local_idx = 0
field_global_idx = host_result['field_indices'][field_local_idx]
start_year = int(allocation.start_year[field_global_idx])

npvs = compute_field_npvs(
    host_result['oil'][field_local_idx],
    host_result['gas'][field_local_idx],
    prices.oil, prices.gas,
    capex=capex_matrix[field_global_idx, plot_host],
    opex_rate=cost_data.opex_rate[plot_host],
    variable_opex_rate=cost_data.variable_opex_rate[plot_host],
    abex=cost_data.abex[plot_host],
    discount_rate=v['discount_rate'],
    start_year=start_year,
    nok_per_usd=v['nok_per_usd'],
    barrel=v['barrel'],
    btu=v['btu'],
)

print(f"NPV array shape: {npvs.shape} (scenarios × years)")
print(f"\nMean NPV by exercise year (MNOK):")
for t in range(config['n_years']):
    vals = npvs[:, t]
    if np.isfinite(vals).any():
        print(f"  Year {t:2d}: mean={vals.mean():10.0f}  std={vals.std():8.0f}  "
              f"min={vals.min():10.0f}  max={vals.max():10.0f}")

In [ ]:
# NPV distribution at a few key exercise years
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_years = [start_year, start_year + 3, start_year + 7]
plot_years = [y for y in plot_years if y < config['n_years']]

for ax, t in zip(axes, plot_years):
    vals = npvs[:, t]
    if np.isfinite(vals).any():
        ax.hist(vals[np.isfinite(vals)], bins=30, edgecolor='black', alpha=0.7)
        ax.axvline(vals.mean(), color='red', linestyle='--', label=f'Mean: {vals.mean():.0f}')
        ax.axvline(0, color='black', linestyle='-', alpha=0.5)
        ax.set_xlabel('NPV (MNOK)')
        ax.set_ylabel('Count')
        ax.set_title(f'Exercise year {t}')
        ax.legend()

plt.suptitle(f'NPV Distribution — {field_data.names[field_global_idx]} on {host_data.names[plot_host]}', y=1.02)
plt.tight_layout()
plt.show()

## 4. LSMC — Exercise timing and ROV

In [ ]:
rov, exercise_years = lsmc_field(
    npvs, prices.oil, prices.gas,
    discount_rate=v['discount_rate'],
    start_year=start_year,
)

print(f"=== LSMC Results for {field_data.names[field_global_idx]} on {host_data.names[plot_host]} ===")
print(f"Mean ROV: {rov.mean():.0f} MNOK")
print(f"Std ROV:  {rov.std():.0f} MNOK")
print(f"Min ROV:  {rov.min():.0f} MNOK")
print(f"Max ROV:  {rov.max():.0f} MNOK")
print(f"\nExercise year distribution:")
for yr in range(-1, config['n_years']):
    count = (exercise_years == yr).sum()
    if count > 0:
        label = 'Never' if yr == -1 else f'Year {yr}'
        print(f"  {label}: {count} paths ({count/len(exercise_years)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Exercise year histogram
ax = axes[0]
exercised = exercise_years[exercise_years >= 0]
if len(exercised) > 0:
    bins = np.arange(exercised.min() - 0.5, exercised.max() + 1.5, 1)
    ax.hist(exercised, bins=bins, edgecolor='black', alpha=0.7, color='steelblue')
never_count = (exercise_years == -1).sum()
ax.set_xlabel('Exercise Year')
ax.set_ylabel('Number of Scenarios')
ax.set_title(f'When does LSMC exercise? (Never: {never_count})')
ax.grid(True, alpha=0.3)

# ROV distribution
ax = axes[1]
ax.hist(rov, bins=30, edgecolor='black', alpha=0.7, color='seagreen')
ax.axvline(rov.mean(), color='red', linestyle='--', lw=2, label=f'Mean: {rov.mean():.0f}')
ax.axvline(0, color='black', linestyle='-', alpha=0.5)
ax.set_xlabel('ROV (MNOK)')
ax.set_ylabel('Count')
ax.set_title('Real Option Value Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'{field_data.names[field_global_idx]} on {host_data.names[plot_host]}', y=1.02)
plt.tight_layout()
plt.show()

## 5. ROV vs Static NPV — Does optionality add value?

Compare:
- **Static NPV**: exercise immediately at `start_year` (no optionality)
- **ROV**: optimal exercise timing from LSMC

ROV should always be >= static NPV because the option to wait can never reduce value.

In [ ]:
# Static value: exercise at start_year, but apply go/no-go (max 0)
static_npv = np.maximum(npvs[:, start_year], 0.0)

# Discount static NPV to year 0 for comparison
static_npv_t0 = static_npv * np.exp(-v['discount_rate'] * start_year)

print(f"{'Metric':<30} {'Static NPV':>12} {'ROV':>12} {'Difference':>12}")
print('-' * 68)
print(f"{'Mean (MNOK)':<30} {static_npv_t0.mean():>12.0f} {rov.mean():>12.0f} {rov.mean() - static_npv_t0.mean():>+12.0f}")
print(f"{'Std (MNOK)':<30} {static_npv_t0.std():>12.0f} {rov.std():>12.0f}")
print(f"{'P5 (MNOK)':<30} {np.percentile(static_npv_t0, 5):>12.0f} {np.percentile(rov, 5):>12.0f}")
print(f"{'P50 (MNOK)':<30} {np.percentile(static_npv_t0, 50):>12.0f} {np.percentile(rov, 50):>12.0f}")
print(f"{'P95 (MNOK)':<30} {np.percentile(static_npv_t0, 95):>12.0f} {np.percentile(rov, 95):>12.0f}")
print(f"{'Zero-value paths':<30} {(static_npv_t0 == 0).sum():>12d} {(rov == 0).sum():>12d}")
print(f"\nOptionality premium: {(rov.mean() / static_npv_t0.mean() - 1) * 100:.2f}%")

## 6. All fields — Full portfolio valuation

Run the valuation for all fields in the test allocation.

In [ ]:
print(f"{'Field':<10} {'Host':<8} {'Start':>5} {'Mean NPV':>10} {'Mean ROV':>10} {'Premium':>8} {'Exercise':>12}")
print('-' * 75)

total_rov = np.zeros(prices.n_scenarios)

for host_idx_loop, host_result_loop in capped.items():
    for local_i, global_i in enumerate(host_result_loop['field_indices']):
        sy = int(allocation.start_year[global_i])
        
        field_npvs = compute_field_npvs(
            host_result_loop['oil'][local_i],
            host_result_loop['gas'][local_i],
            prices.oil, prices.gas,
            capex=capex_matrix[global_i, host_idx_loop],
            opex_rate=cost_data.opex_rate[host_idx_loop],
            variable_opex_rate=cost_data.variable_opex_rate[host_idx_loop],
            abex=cost_data.abex[host_idx_loop],
            discount_rate=v['discount_rate'],
            start_year=sy,
            nok_per_usd=v['nok_per_usd'],
            barrel=v['barrel'],
            btu=v['btu'],
        )
        
        field_rov, field_ex = lsmc_field(
            field_npvs, prices.oil, prices.gas,
            discount_rate=v['discount_rate'],
            start_year=sy,
        )
        
        total_rov += field_rov
        
        static = np.maximum(field_npvs[:, sy], 0.0) * np.exp(-v['discount_rate'] * sy)
        premium = (field_rov.mean() / static.mean() - 1) * 100 if static.mean() > 0 else float('nan')
        
        # Most common exercise year
        exercised = field_ex[field_ex >= 0]
        if len(exercised) > 0:
            mode_yr = int(np.bincount(exercised.astype(int)).argmax())
            ex_str = f"yr {mode_yr} ({len(exercised)/len(field_ex)*100:.0f}%)"
        else:
            ex_str = "never"
        
        print(f"{field_data.names[global_i]:<10} {host_data.names[host_idx_loop]:<8} "
              f"{sy:>5d} {static.mean():>10.0f} {field_rov.mean():>10.0f} "
              f"{premium:>7.2f}% {ex_str:>12}")

print(f"\nPortfolio total ROV: mean={total_rov.mean():.0f}, std={total_rov.std():.0f} MNOK")

## 7. Discount rate sensitivity

How does the discount rate affect ROV? Higher rates reduce present value
but increase the value of the option to wait.

In [ ]:
# Sensitivity for the first field on the selected host
host_result_sens = capped[plot_host]
field_local_sens = 0
field_global_sens = host_result_sens['field_indices'][field_local_sens]
start_yr = int(allocation.start_year[field_global_sens])

rates = [0.04, 0.06, 0.08, 0.10, 0.12, 0.15]
mean_rovs = []
mean_statics = []

for r in rates:
    npv_r = compute_field_npvs(
        host_result_sens['oil'][field_local_sens],
        host_result_sens['gas'][field_local_sens],
        prices.oil, prices.gas,
        capex=capex_matrix[field_global_sens, plot_host],
        opex_rate=cost_data.opex_rate[plot_host],
        variable_opex_rate=cost_data.variable_opex_rate[plot_host],
        abex=cost_data.abex[plot_host],
        discount_rate=r,
        start_year=start_yr,
        nok_per_usd=v['nok_per_usd'],
        barrel=v['barrel'],
        btu=v['btu'],
    )
    rov_r, _ = lsmc_field(
        npv_r, prices.oil, prices.gas,
        discount_rate=r, start_year=start_yr,
    )
    static_r = np.maximum(npv_r[:, start_yr], 0.0) * np.exp(-r * start_yr)
    mean_rovs.append(rov_r.mean())
    mean_statics.append(static_r.mean())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rates, mean_statics, 'o--', label='Static NPV (exercise immediately)', color='gray')
ax.plot(rates, mean_rovs, 's-', label='ROV (LSMC optimal exercise)', color='steelblue')
ax.set_xlabel('Discount Rate')
ax.set_ylabel('Mean Value (MNOK)')
ax.set_title(f'Discount Rate Sensitivity — {field_data.names[field_global_sens]}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()